# 02 · Análisis exploratorio

**Hipótesis de negocio.** La potencia que una planta *debería* generar se puede estimar con el clima
(irradiancia y temperatura) con un error chico. Si el error típico del modelo es bastante menor que
las pérdidas que nos interesan, las horas o meses en que la generación real queda por debajo de la
esperada permiten detectar fallas (caídas abruptas) y suciedad (*soiling*, pérdida gradual) y
cuantificar la energía perdida.

**Cómo se mide.** (1) Error del modelo de potencia esperada en meses no vistos (MAE, RMSE, R²).
(2) Sobre fallas sintéticas inyectadas: precisión, *recall* y demora de detección. La hipótesis se
sostiene si el detector encuentra caídas de 10% o más con pocas falsas alarmas.

Este notebook revisa si los datos permiten esa estimación y qué hay que limpiar antes.
Los gráficos marcados con ⭐ son los propuestos para la defensa.

In [ ]:
# Si corrés en Colab: clonar el repo (una vez por sesión), pararse en él e instalar dependencias
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/tpo-solar'):
        !git clone https://github.com/imelinc/tpo-solar.git /content/tpo-solar
    %cd /content/tpo-solar
    !pip install -q -r requirements.txt
sys.path.append('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
from src.config import DATA_RAW, DATA_PROCESSED, MODELS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import agregar_features_solares, limpiar_pvod, filtro_rendimiento_argentina

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
A = pd.read_parquet(DATA_PROCESSED / "dataset_A_pvod_hourly.parquet")
na_originales = int(A.isna().sum().sum())
B = pd.read_parquet(DATA_PROCESSED / "dataset_B_argentina_mensual.parquet")
A = agregar_features_solares(A)                  # POA, cielo claro y temp. de celda con Open-Meteo
A["rend_rel"] = A["pvod_power_norm"] / (A["om_poa"] / 1000)  # potencia por kW/m² en el plano del panel
dia = A[A["is_daylight"] == 1]
print(A.shape, B.shape)

## A · Global (PVOD, 10 plantas en China)

### A.1 Cobertura y faltantes

In [ ]:
def cobertura(x):
    dias = pd.Series(sorted(pd.to_datetime(x["date_local"].unique())))
    saltos = dias.diff().dt.days
    return pd.Series({"desde": dias.min().date(), "hasta": dias.max().date(), "dias": len(dias),
                      "dias_faltantes": int((saltos - 1).clip(lower=0).sum()),
                      "huecos": int((saltos > 1).sum()), "cap_mw": x["plant_capacity_mw"].iloc[0]})

cob = A.groupby("plant_id").apply(cobertura, include_groups=False)
print(cob)
print("\nValores vacíos en las columnas originales del dataset horario:", na_originales)

**Conclusión:** Seis plantas cubren casi un año (jul-2018 a jun-2019). station00 y station09 arrancan más tarde, station03 tiene 5 meses y station05 solo 3. Faltan pocos días (como mucho 11, en station06) y no hay valores vacíos. Para el modelo: validar dejando meses afuera y comparar plantas solo en el período común (mar-jun 2019 incluye a las 10).

### A.2 Distribución de la potencia por planta

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=dia, x="plant_id", y="pvod_power_norm", ax=ax, fliersize=1, color="#9ecae1")
ax.axhline(1, color="red", ls="--", lw=1, label="capacidad declarada")
ax.set(xlabel="", ylabel="potencia / capacidad", title="Potencia normalizada en horas de sol")
ax.legend(); plt.show()
print(dia.groupby("plant_id")["pvod_power_norm"].quantile([0.5, 0.99, 1]).unstack().round(2))

**Conclusión:** La mediana de potencia en horas de sol va del 12% (station09) al 41% (station04) de la capacidad. Dos plantas se salen del patrón: station04 supera su capacidad declarada (máximo de 1,28 en el promedio horario) y station09 nunca pasa de 0,57. Si no se corrige, comparar plantas con la potencia normalizada lleva a conclusiones equivocadas.

### A.3 ⭐ Potencia vs. irradiancia: ¿se puede predecir la generación con el clima?

In [ ]:
fuentes = {"Sensor de planta (LMD)": "pvod_lmd_totalirrad", "Open-Meteo GHI": "om_shortwave_radiation",
           "Open-Meteo en plano del panel": "om_poa", "NASA POWER GHI": "nasa_ALLSKY_SFC_SW_DWN",
           "Pronóstico NWP": "pvod_nwp_globalirrad"}
corr = pd.DataFrame({k: dia.groupby("plant_id").apply(lambda x: x["pvod_power_norm"].corr(x[v]), include_groups=False)
                     for k, v in fuentes.items()})
print("Correlación (Pearson) potencia vs. irradiancia, horas de sol:")
print(corr.round(3)); print("\nMediana entre plantas:"); print(corr.median().round(3))

ref = dia[~dia["plant_id"].isin(["station04", "station09"])]
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, col, tit in zip(axs, ["pvod_lmd_totalirrad", "om_poa"], ["Sensor de planta", "Open-Meteo (satélite/reanálisis) en plano del panel"]):
    ax.hexbin(ref[col], ref["pvod_power_norm"], gridsize=60, bins="log", cmap="viridis", mincnt=1)
    ax.set(xlabel="irradiancia (W/m²)", title=tit, xlim=(0, 1300))
axs[0].set_ylabel("potencia / capacidad")
fig.suptitle("8 plantas (sin station04 ni station09)"); plt.tight_layout(); plt.show()

**Conclusión:** La irradiancia explica casi toda la generación. La correlación mediana es de 0,965 con el sensor de planta y de 0,91-0,92 con fuentes externas gratuitas (Open-Meteo y NASA). El pronóstico NWP queda atrás (0,85). Para el negocio: con el sensor de planta se estima bien la potencia esperada, y cuando el sensor falla o no existe (como en Argentina) el clima satelital sirve de respaldo, con algo más de error. **Esto sostiene la primera parte de la hipótesis.**

### A.4 Efecto de la temperatura

In [ ]:
alta = dia[(dia["om_poa"] > 600) & dia["rend_rel"].between(0.05, 2)
           & ~dia["plant_id"].isin(["station04", "station09"])].copy()
alta["temp_bin"] = pd.cut(alta["om_temp_celda"], range(-10, 75, 5))
t = alta.groupby(["temp_bin"], observed=True)["rend_rel"].agg(["median", "size"])
print(t.round(3))
# Pendiente dentro de cada planta y mes para separar temperatura de estación
alta["anom_t"] = alta["om_temp_celda"] - alta.groupby(["plant_id", "month"])["om_temp_celda"].transform("mean")
alta["anom_r"] = alta["rend_rel"] / alta.groupby(["plant_id", "month"])["rend_rel"].transform("median") - 1
pend = np.polyfit(alta["anom_t"], alta["anom_r"], 1)[0]
print(f"\nPendiente dentro de planta×mes: {pend*100:.2f} % de rendimiento por °C de celda")

fig, ax = plt.subplots(figsize=(8, 4))
mid = [iv.mid for iv in t.index]
ax.plot(mid, t["median"], marker="o")
ax.set(xlabel="temperatura de celda estimada (°C)", ylabel="potencia por kW/m² en plano",
       title="Rendimiento con irradiancia alta (> 600 W/m²) según temperatura")
plt.show()

**Conclusión:** En estos datos no se ve el efecto de la temperatura. Dentro de cada planta y mes, la pendiente es prácticamente nula (+0,05% por °C), aunque la física de los módulos indicaría que el calor resta rendimiento. En la mirada general el rendimiento hasta sube con el calor, pero eso es la estación: en invierno (frío) las pérdidas son más grandes (ver A.6). Además, la temperatura de celda se estima con clima de grilla y no se mide. Para el modelo: la temperatura entra como feature, pero el peso lo llevan la irradiancia y la época del año.

### A.5 ⭐ Perfiles diarios y estacionales

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
perfil = ref.pivot_table(index="hour_local", columns="month", values="pvod_power_norm", aggfunc="mean")
sns.heatmap(perfil, cmap="YlOrRd", ax=axs[0], cbar_kws={"label": "potencia / capacidad"})
axs[0].set(title="Potencia media por hora local y mes", xlabel="mes", ylabel="hora local")
ener = (A.groupby(["plant_id", "month"])["pvod_energy_mwh"].sum()
        / A.groupby(["plant_id", "month"])["date_local"].nunique()
        / A.groupby(["plant_id", "month"])["plant_capacity_mw"].first()).unstack(0)
ener.plot(ax=axs[1], marker=".")
axs[1].set(title="Energía diaria por MW instalado", xlabel="mes", ylabel="MWh / MW / día")
axs[1].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
print(ener.round(2))

**Conclusión:** Las plantas generan entre las 7 y las 17 h, con el pico entre las 11 y las 13 h. El mejor mes es mayo, no junio ni julio: en julio y agosto hay más nubes y la generación baja. De noviembre a febrero se genera la mitad o menos. Para el negocio: comparar contra el mes anterior no alcanza. La referencia tiene que ajustarse por hora, estación y clima del día.

### A.6 ⭐ Comparación entre plantas: rendimiento relativo por mes

`rend_rel` = potencia normalizada / (irradiancia en el plano del panel / 1000). Una planta sana debería dar un valor parecido todo el año; las caídas son pérdidas candidatas.

In [ ]:
rr = dia[dia["om_poa"] > 500].pivot_table(index="plant_id", columns="month", values="rend_rel", aggfunc="median")
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(rr, annot=True, fmt=".2f", cmap="RdYlGn", center=0.85, ax=ax, cbar_kws={"label": "rend_rel"})
ax.set(title="Rendimiento relativo mediano (horas con > 500 W/m²)", xlabel="mes", ylabel="")
plt.show()

**Conclusión:** Es el gráfico clave para la hipótesis: muestra cuánto genera cada planta por unidad de sol, mes a mes. Hay tres patrones. (1) station01, 04, 05 y 08 se mantienen estables todo el año, como se espera de una planta sana. (2) station00 y station06 pierden mucho en invierno: station06 cae de 0,84 en julio a 0,27 en enero. Esto es compatible con sombras entre filas por el sol bajo o con nieve, pero falta confirmarlo. (3) station09 rinde la mitad que el resto en todo el período. El modelo de potencia esperada tiene que aprender lo estacional de cada planta para no disparar falsas alarmas cada invierno.

### A.7 Rarezas

**station04 y station09: ¿error de metadata o caso real?**

In [ ]:
meta = pd.read_csv(DATA_PROCESSED / "pvod_plantas_metadata.csv")
meta["wp_modulo"] = meta["plant_module_info"].str.extract(r"Pmax:\s*(\d+)").astype(float)
meta["mwp_dc_modulos"] = meta["plant_panel_number"] * meta["wp_modulo"] / 1e6
meta = meta.set_index("plant_id")[["plant_capacity_mw", "wp_modulo", "plant_panel_number", "mwp_dc_modulos"]]
meta["p_max_mw"] = A.groupby("plant_id")["pvod_power"].max()
meta["p_max/dc"] = meta["p_max_mw"] / meta["mwp_dc_modulos"]
# Sensor de planta vs. cielo claro horizontal al mediodía: > 1 sostenido = sensor inclinado o descalibrado
kt = A[A["hour_local"].between(11, 13)].groupby(["plant_id", "month"])["pvod_lmd_totalirrad"].quantile(0.9)
cs = A[A["hour_local"].between(11, 13)].groupby(["plant_id", "month"])["om_shortwave_radiation"].quantile(0.9)
meta["lmd/om_invierno"] = (kt / cs).unstack().loc[:, [11, 12, 1]].mean(axis=1)
meta["lmd/om_verano"] = (kt / cs).unstack().loc[:, [6, 7]].mean(axis=1)
print(meta.round(2))
s9 = A[A["plant_id"] == "station09"].groupby("date_local").agg(
    p_max=("pvod_power", "max"), poa_om_max=("om_poa", "max"), sensor_max=("pvod_lmd_totalirrad", "max"))
s9["rend_dia"] = s9["p_max"] / (s9["poa_om_max"] / 1000) / 20
print("\nstation09, últimas semanas (rendimiento diario y sensor de planta):")
print(s9.loc["2019-05-10":].round(2).T)

**Conclusión:** **station04:** con 75.680 módulos de 265 Wp, la potencia de los paneles es de 20,06 MWp, pero la planta llega a 25,69 MW (1,28 veces). Con esa ficha es físicamente imposible. Además, su sensor lee de más: de 1,45 a 1,89 veces el GHI externo, con 11 horas por encima de 1400 W/m². Conclusión: hay un error de metadata o de escala de medición. Se puede modelar con su propia historia, pero no compararla en valores absolutos.

**station09:** tiene 22,4 MWp de módulos y no pasa de 11,5 MW (0,51). No parece un error fijo de metadata. Desde mediados de mayo de 2019 su rendimiento diario sube en escalones (de ~0,40 a ~0,49, ~0,55, ~0,62 y ~0,67), lo que es compatible con bloques de la planta que vuelven a servicio. Es el mejor ejemplo real de bajo rendimiento sostenido del dataset. Además, el sensor de irradiancia de la planta deja de funcionar el 4/6/2019 (lee menos de 30 W/m²), lo que es otro motivo para tener clima externo de respaldo.

**Sensor de planta en general:** en invierno lee bastante más que el GHI horizontal, sobre todo en station02, 03, 04, 06, 08 y 09. Probablemente está inclinado como los paneles, así que **no se le aplica la transposición de pvlib**: se usa directo como irradiancia en el plano del panel.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
for p, col in [("station08", "grey"), ("station04", "tab:red"), ("station09", "tab:blue")]:
    x = dia[(dia["plant_id"] == p) & (dia["om_poa"] > 0)]
    x = x.sample(min(3000, len(x)), random_state=0)
    axs[0].scatter(x["om_poa"], x["pvod_power_norm"], s=3, alpha=0.4, color=col, label=p)
axs[0].set(xlabel="irradiancia en plano, Open-Meteo (W/m²)", ylabel="potencia / capacidad",
           title="station04 genera de más, station09 de menos"); axs[0].legend(markerscale=4)
(kt / cs).unstack(0)[["station01", "station04", "station08", "station09"]].plot(ax=axs[1], marker=".")
axs[1].axhline(1, color="k", lw=0.8)
axs[1].set(xlabel="mes", ylabel="sensor planta / GHI Open-Meteo (p90 mediodía)",
           title="El sensor de planta lee más que el GHI en invierno")
ev = A[(A["plant_id"] == "station01") & A["date_local"].astype(str).between("2018-12-07", "2018-12-10")]
axs[2].plot(ev["timestamp_local_utc8"], ev["pvod_power"], label="potencia (MW)")
axs[2].plot(ev["timestamp_local_utc8"], ev["pvod_lmd_totalirrad"] / 30, label="irradiancia planta / 30")
axs[2].set(title="station01: dato congelado 8-9/12/2018"); axs[2].legend(); axs[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

**station01: potencia nocturna**

In [ ]:
noche = A[A["is_daylight"] == 0]
print("Potencia nocturna media por planta (MW):"); print(noche.groupby("plant_id")["pvod_power"].mean().round(3))
L = limpiar_pvod(A)
print("\nHoras con dato congelado (potencia idéntica >= 3 h):"); print(L.groupby("plant_id")["dato_congelado"].sum())
sin_evento = noche[~L.loc[noche.index, "dato_congelado"]]
print("\nstation01, potencia nocturna media sin las horas congeladas:",
      round(sin_evento.loc[sin_evento["plant_id"] == "station01", "pvod_power"].mean(), 4), "MW")

**Conclusión:** La potencia nocturna de station01 no es ruido del medidor: es un **dato congelado**. Desde el 8/12/2018 a las 14 h hasta el 9/12 a las 13 h, el registrador repite 15,52 MW y 452 W/m² durante 24 horas. Sin esas horas, la potencia nocturna media baja de 0,059 a 0,004 MW. `limpiar_pvod` marca estos tramos (33 horas en station01, 3 en station02 y 5 en station08) para sacarlos del entrenamiento. Es una falla del sistema de medición, no de la planta, y el reporte tiene que tratarla aparte.

### A.8 Outliers: horas sin generación con sol

In [ ]:
sol = L[(L["is_daylight"] == 1) & (L["om_poa"] > 500)]
apagon = sol[(sol["pvod_power_norm"] < 0.05) & ~sol["dato_congelado"]]
res = pd.DataFrame({"horas_con_sol": sol.groupby("plant_id").size(),
                    "horas_apagado": apagon.groupby("plant_id").size(),
                    "dias_afectados": apagon.groupby("plant_id")["date_local"].nunique()}).fillna(0).astype(int)
res["%_horas"] = (100 * res["horas_apagado"] / res["horas_con_sol"]).round(2)
print(res)
print("\nHoras con irradiancia de planta > 1400 W/m² (físicamente dudoso para GHI):")
print(A[A["pvod_lmd_totalirrad"] > 1400].groupby("plant_id").size())

**Conclusión:** Con mucho sol (más de 500 W/m² en el plano) casi nunca hay generación cero: entre el 0% y el 1,2% de las horas, según la planta, repartidas en 0 a 8 días. Son fallas reales candidatas (inversor parado, corte de red) que el detector debería marcar. station09 es la que más días tiene (8).

## B · Argentina (CAMMESA, 97 parques, mensual)

### B.1 Cobertura y faltantes

In [ ]:
print(B[["potencia_mw", "factor_capacidad", "grv_nombre_parque", "om_ghi_kwh_m2_mes"]].isna().mean().mul(100).round(1))
B["ok"] = filtro_rendimiento_argentina(B)
parques = B.groupby("central").agg(coord=("coord_source", "first"), meses=("fecha", "size"))
print("\nParques con coordenada aproximada (centroide de provincia):",
      (parques["coord"] == "osm_centroide_provincia").sum(), "de", len(parques))
print("Filas aptas para analizar rendimiento:", int(B["ok"].sum()), "de", len(B),
      "| parques:", B.loc[B["ok"], "central"].nunique())
anual = B.groupby("anio").agg(parques=("central", "nunique"), gwh=("energia_gwh", "sum"))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(anual.index, anual["gwh"], color="#fdae6b"); ax.set(ylabel="GWh / año", xlabel="")
ax2 = ax.twinx(); ax2.plot(anual.index, anual["parques"], color="k", marker="o"); ax2.set_ylabel("parques activos")
ax2.grid(False)
ax.set_title("Generación solar en Argentina (2026 llega hasta septiembre)"); plt.show()
print(anual.round(0).astype(int).T)

**Conclusión:** La generación solar en Argentina pasó de 1 parque en 2011 a 97 en 2026, y de 108 GWh en 2018 a 5.106 GWh en 2025. Falta la potencia en el 7% de las filas. Para analizar rendimiento se excluyen los parques con coordenada aproximada (29), las filas sin potencia, con energía 0 o con FC > 0,45, y los parques ampliados fuera de la ventana confiable. Quedan 3.597 filas de 56 parques.

### B.2 ⭐ Factor de capacidad vs. irradiación mensual

In [ ]:
ok = B[B["ok"]]
r_ok = ok["factor_capacidad"].corr(ok["om_ghi_kwh_m2_mes"])
todo = B[B["factor_capacidad"].notna()]
r_todo = todo["factor_capacidad"].corr(todo["om_ghi_kwh_m2_mes"])
print(f"Correlación FC vs. irradiación: filas filtradas {r_ok:.3f} | todas {r_todo:.3f}")
# Dentro de cada parque (quita diferencias de tecnología/seguidores entre parques)
dm = ok.assign(fc=ok["factor_capacidad"] - ok.groupby("central")["factor_capacidad"].transform("mean"),
               g=ok["om_ghi_kwh_m2_mes"] - ok.groupby("central")["om_ghi_kwh_m2_mes"].transform("mean"))
print(f"Correlación dentro de cada parque: {dm['fc'].corr(dm['g']):.3f}")
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=ok, x="om_ghi_kwh_m2_mes", y="factor_capacidad", hue="region", s=12, alpha=0.6, ax=ax)
ax.set(xlabel="irradiación horizontal del mes (kWh/m²)", ylabel="factor de capacidad",
       title="Más sol, más generación (filas filtradas)"); ax.legend(fontsize=7); plt.show()

**Conclusión:** La correlación entre el factor de capacidad y la irradiación del mes es de 0,69, y sube a 0,74 dentro de cada parque. Es más baja que en China porque el dato es mensual, la irradiación es horizontal y la tecnología cambia entre parques (seguidores, inclinación). Igual alcanza para estimar la generación esperada de cada parque y marcar los meses que se apartan.

### B.3 Estacionalidad y temperatura

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=ok, x="mes", y="factor_capacidad", color="#a1d99b", fliersize=1, ax=ax)
ax2 = ax.twinx(); ax2.plot(range(12), ok.groupby("mes")["om_ghi_kwh_m2_mes"].median().values, color="orange", marker="o")
ax2.set_ylabel("irradiación mediana (kWh/m²)", color="orange"); ax2.grid(False)
ax.set(title="Factor de capacidad por mes (hemisferio sur)", xlabel="mes"); plt.show()
print(ok.groupby("estacion_hs")[["factor_capacidad", "performance_ratio_aprox"]].median().round(3))
# Temperatura: anomalía del PR aprox. vs. anomalía de temperatura máxima, dentro de parque y mes calendario
g = ok.groupby(["central", "mes"])
pr_an = ok["performance_ratio_aprox"] / g["performance_ratio_aprox"].transform("median") - 1
t_an = ok["om_temp_max_media_c"] - g["om_temp_max_media_c"].transform("mean")
print(f"\nPR aprox vs. temperatura (dentro de parque y mes): pendiente {np.polyfit(t_an, pr_an, 1)[0]*100:.2f} %/°C, "
      f"r = {pr_an.corr(t_an):.3f}")

**Conclusión:** En verano el FC es de 0,33 y en invierno de 0,19. El PR aproximado sale más alto en invierno (1,18 contra 1,05), pero eso se debe a que usa irradiación horizontal, no a una mejora real. Por eso sirve para comparar parques entre sí y no para medir eficiencia absoluta. En el dato mensual la temperatura no muestra efecto (r = 0,03).

### B.4 Comparación entre parques

In [ ]:
pr = (ok.groupby(["central", "region"])["performance_ratio_aprox"].agg(["median", "size"])
        .query("size >= 12").sort_values("median").reset_index())
fig, ax = plt.subplots(figsize=(14, 4))
sns.barplot(data=pr, x="central", y="median", hue="region", dodge=False, ax=ax)
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.set(xlabel="", ylabel="PR aprox. mediano", title="PR aproximado por parque (≥ 12 meses válidos). Solo sirve para comparar")
ax.legend(fontsize=7, ncol=4); plt.show()
print("Parques:", len(pr), "| rango de PR mediano:", pr["median"].min().round(2), "a", pr["median"].max().round(2))
print(pr.head(5).round(2)); print(pr.tail(5).round(2))

**Conclusión:** El PR aproximado mediano va de 0,44 (VMRSFV) a 1,37 (TAMBFV). Los de arriba son de Cuyo y probablemente tienen seguidores (falta confirmarlo). Los de abajo (VMRSFV y VMR2FV, con 18 meses válidos) rinden menos de la mitad de lo típico: o tienen un problema real o la potencia o la coordenada están mal cargadas. Son los primeros candidatos para revisar con el área técnica.

### B.5 Rarezas

In [ ]:
B = B.sort_values(["central", "fecha"])
primero = B[B["energia_mwh"] > 0].groupby("central")["fecha"].min()
ultimo = B[B["energia_mwh"] > 0].groupby("central")["fecha"].max()
z = B[B["energia_mwh"] == 0].join(primero.rename("primero"), on="central").join(ultimo.rename("ultimo"), on="central")
z["tipo"] = np.select([z["primero"].isna(), z["fecha"] < z["primero"], z["fecha"] > z["ultimo"]],
                      ["parque sin ningún mes con generación", "antes de empezar a generar", "después de dejar de generar"],
                      "en el medio (parada o falla)")
print("Meses con energía 0:", len(z)); print(z["tipo"].value_counts())
print("\nParques con meses en 0 'en el medio':", z.loc[z["tipo"] == "en el medio (parada o falla)", "central"].value_counts().to_dict())
alto = B[B["factor_capacidad"] > 0.45]
print("\nFC > 0,45:", len(alto), "filas"); print(alto.groupby("central").agg(meses=("fecha", "size"), fc_max=("factor_capacidad", "max"),
      fuente_potencia=("potencia_source", lambda s: s.str.startswith("GRV_mismo").mean())).round(2))

**Conclusión:** De los 208 meses con energía 0, 173 son de parques que nunca generaron en la base y 15 son anteriores al arranque. Solo 20 caen en medio de la vida del parque (UL42FV 10, ATI1FV 4, CUM2FV 3, y uno cada uno en ALT1FV, CHEPFV y TINZFV): esas son las paradas reales candidatas. Las 35 filas con FC > 0,45 se concentran en CHI1FV, HON1FV y TAMBFV. En CHI1FV y HON1FV se usa la potencia del mismo mes, así que no se explican por tomar la de un mes cercano. Lo más probable es que la potencia habilitada sea AC y menor que la DC, o que haya seguidores. Se excluyen del entrenamiento, pero no se borran.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3.5), sharex=True)
for ax, p in zip(axs, ["CUM2FV", "PINFFV", "VANGFV"]):
    x = B[B["central"] == p]; f = pd.to_datetime(x["fecha"])
    ax.plot(f, x["factor_capacidad"], marker=".", label="FC calculado")
    ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2025-09-01"), color="green", alpha=0.1, label="potencia del mismo mes")
    ax2 = ax.twinx(); ax2.step(f, x["potencia_mw"], color="grey", where="post"); ax2.set_ylabel("MW"); ax2.grid(False)
    ax.set_title(p); ax.tick_params(axis="x", rotation=30)
axs[0].legend(fontsize=7); axs[0].set_ylabel("factor de capacidad"); plt.tight_layout(); plt.show()

**Conclusión:** En los tres parques ampliados, el FC calculado se ve coherente con la potencia de cada etapa. CUM2FV con 4 MW antes de 2022 tiene un ciclo anual normal. El riesgo real está en los meses de arranque de cada ampliación, cuando la potencia cambia a mitad de mes (FC muy bajo al inicio de PINFFV y VANGFV). El filtro conservador igual los deja solo dentro de la ventana 2022-01 a 2025-09.

### B.6 Outliers dentro de cada parque: un primer detector simple

Para cada parque y mes: z robusto del PR aprox. respecto de la mediana del propio parque (mediana y MAD). Es una vista previa de `04_deteccion`, no el detector final.

In [ ]:
gp = ok.groupby("central")["performance_ratio_aprox"]
mad = gp.transform(lambda s: (s - s.median()).abs().median())
zr = (ok["performance_ratio_aprox"] - gp.transform("median")) / (1.4826 * mad)
bajos = ok[zr < -3]
print(f"Meses con PR muy por debajo de lo normal del parque (z < -3): {len(bajos)} de {len(ok)} "
      f"({100*len(bajos)/len(ok):.1f}%), en {bajos['central'].nunique()} parques")
print(bajos["central"].value_counts().head(8))
ej = bajos["central"].value_counts().index[0]
x = ok[ok["central"] == ej]; f = pd.to_datetime(x["fecha"])
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(f, x["performance_ratio_aprox"], marker=".")
ax.scatter(pd.to_datetime(bajos.loc[bajos["central"] == ej, "fecha"]),
           bajos.loc[bajos["central"] == ej, "performance_ratio_aprox"], color="red", zorder=3, label="z < -3")
ax.set(title=f"{ej}: meses con rendimiento anómalo", ylabel="PR aprox."); ax.legend(); plt.show()

**Conclusión:** Un detector simple (z robusto del PR contra la historia del propio parque) marca 155 meses (4,3%) en 47 parques. En el ejemplo de ULL4FV, todo 2025 tiene un PR cerca de 0,15 cuando lo normal es ~1,1: casi un año de pérdida casi total, compatible con una falla prolongada o con un cambio de potencia no registrado. También marca los primeros meses de operación (arranque), que hay que excluir. Esto es lo que `04_deteccion` tiene que mejorar: menos falsas alarmas y una estimación de la energía perdida en MWh.

## Resumen para la defensa

Proponemos contar la historia en cuatro gráficos:

1. **A.3 – El clima predice la generación** (correlación de 0,965 con el sensor y de ~0,91 con el satélite). Base de la hipótesis.
2. **A.6 – Rendimiento relativo por planta y mes.** Las plantas sanas se ven planas; las pérdidas estacionales (station06) y crónicas (station09) saltan a la vista.
3. **A.7 – Rarezas que son fallas reales:** station09 se recupera en escalones (caso real de bajo rendimiento), station01 tiene un dato congelado (falla de medición) y station04 muestra un error de ficha técnica (por qué no se comparan plantas en valores absolutos).
4. **B.6 – Argentina:** ULL4FV pierde casi toda la generación durante 2025, y un detector simple ya la encuentra.

**Decisiones para el modelado:** usar Open-Meteo como fuente de irradiancia en común para China y Argentina (el sensor de planta como variante); no transponer el sensor de planta; entrenar sin datos congelados, apagones ni meses de arranque; normalizar por la historia de cada planta y no por la capacidad declarada.